## Carga de datos y Limpieza

In [71]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# Carga 
df = pd.read_csv('data.csv').dropna(subset=['Primary streaming service', 'Fav genre', 'Music effects'])
espanol = {'Improve': 'Mejora', 'Worsen': 'Empeora', 'No effect': 'Sin efecto', 'I do not use a streaming service.': 'Ninguna'}
df = df.replace(espanol)


## Organización

In [72]:
# Agrupación por categorías
p_top = df['Primary streaming service'].value_counts().nlargest(3).index
g_top = df['Fav genre'].value_counts().nlargest(6).index

df['Plat'] = np.where(df['Primary streaming service'].isin(p_top), df['Primary streaming service'], 'Otras Plataformas')
df['Gen'] = np.where(df['Fav genre'].isin(g_top), df['Fav genre'], 'Otros Géneros')

# Flujos
f1 = df.groupby(['Plat', 'Gen']).size().reset_index()
f2 = df.groupby(['Gen', 'Music effects']).size().reset_index()
f1.columns = f2.columns = ['origen', 'destino', 'valor']
links = pd.concat([f1, f2], ignore_index=True)

## Nodos

In [ ]:
nodos = list(pd.unique(links[['origen', 'destino']].values.ravel('F')))
links['id_origen'] = links['origen'].apply(nodos.index)
links['id_destino'] = links['destino'].apply(nodos.index)

## Color

In [74]:
colores = (px.colors.qualitative.Pastel + px.colors.qualitative.Set3)
color_nodos = colores[:len(nodos)]

arr_nodos = np.array(color_nodos)
arr_rgba = np.char.replace(arr_nodos, 'rgb', 'rgba')
arr_rgba = np.char.replace(arr_rgba, ')', ', 0.35)')

color_links = arr_rgba[links['id_origen'].values]

## Grafico

In [ ]:
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(pad=20, thickness=25, line=dict(color="black", width=0.5), label=nodos, color=color_nodos),
    link=dict(source=links['id_origen'], target=links['id_destino'], value=links['valor'], color=color_links)
))

fig.update_layout(
    title_text="Plataforma -> Género Favorito -> Efecto en Salud Mental",
    paper_bgcolor="white", plot_bgcolor="white", font_color="black", height=700
)

fig.show()